In [ ]:
# !pip install ipywidgets pillow

In [ ]:
# Simple Jupyter Notebook frontend for mock road sign classification
# Required packages:
# !pip install ipywidgets pillow

import random
from io import BytesIO

import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image


ROAD_SIGN_CLASSES = [
    "Stop",
    "Speed Limit",
    "No Entry",
    "Yield",
    "Pedestrian Crossing",
    "Priority Road",
    "Go Straight",
    "Road Works",
    "Dangerous Curve",
    "Parking"
]


def fake_model_predict(image):
    """
    Temporary placeholder instead of a real AI model.

    The classification is currently fully random.
    In the future, a real CNN/PyTorch model can be connected here.
    """
    predicted_class = random.choice(ROAD_SIGN_CLASSES)
    confidence = random.uniform(0.55, 0.99)
    return predicted_class, confidence


upload_widget = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload image"
)

classify_button = widgets.Button(
    description="Classify sign",
    button_style="success"
)

reset_button = widgets.Button(
    description="Clear",
    button_style="warning"
)

image_output = widgets.Output()
result_output = widgets.Output()


def get_uploaded_image():
    """
    Reads the uploaded image from the FileUpload widget.
    Supports different ipywidgets versions.
    """
    if not upload_widget.value:
        return None

    uploaded = upload_widget.value

    if isinstance(uploaded, tuple):
        file_info = uploaded[0]
        content = file_info["content"]

    elif isinstance(uploaded, list):
        file_info = uploaded[0]
        content = file_info["content"]

    elif isinstance(uploaded, dict):
        file_info = next(iter(uploaded.values()))
        content = file_info["content"]

    else:
        return None

    return Image.open(BytesIO(content)).convert("RGB")


def show_uploaded_image(change=None):
    """
    Displays a preview of the uploaded image.
    """
    with image_output:
        clear_output(wait=True)

        image = get_uploaded_image()

        if image is None:
            print("No image has been uploaded yet.")
            return

        preview = image.copy()
        preview.thumbnail((500, 500))

        print("Uploaded image preview:")
        display(preview)


def classify_image(button):
    """
    Runs random road sign classification.
    """
    with result_output:
        clear_output(wait=True)

        image = get_uploaded_image()

        if image is None:
            print("Please upload a road sign image first.")
            return

        predicted_class, confidence = fake_model_predict(image)

        print("Classification result:")
        print(f"Predicted class: {predicted_class}")
        print(f"Model confidence: {confidence:.2%}")
        print()
        print("Note: this result is random because the real model is not connected yet.")


def reset_app(button):
    """
    Clears the displayed image preview and classification result.

    The uploaded file itself is not reset because FileUpload.value is read-only
    in some notebook environments, including Google Colab.
    To upload a different image, use the upload button again.
    """
    with image_output:
        clear_output(wait=True)
        print("Image preview cleared.")

    with result_output:
        clear_output(wait=True)
        print("Classification result cleared.")


upload_widget.observe(show_uploaded_image, names="value")
classify_button.on_click(classify_image)
reset_button.on_click(reset_app)


title = widgets.HTML(
    """
    <h2>Road Sign Recognition Application</h2>

    <p>
        <b>Authors:</b> Sylwia Rybak, Wojciech Sendek, Stanisław Zieliński, Jakub Szostak
    </p>

    <p>
        A simple prototype interface for road sign classification.
        The classification is currently random because the AI model has not been implemented yet.
    </p>
    """
)

controls = widgets.HBox([
    upload_widget,
    classify_button,
    reset_button
])

app = widgets.VBox([
    title,
    controls,
    image_output,
    result_output
])

display(app)